In [3]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Dataset Ampliado para Entrenamiento Robusto
np.random.seed(42)
n_samples = 200

clients = [f'CLI-{10000+i}' for i in range(15)]
client_profiles = {
    c: {
        'ingreso': np.random.uniform(12000, 55000),
        'ahorro': np.random.uniform(0, 45000),
        'score': np.random.randint(480, 850),
    }
    for c in clients
}

categories = [
    'Supermercado',
    'Comida rápida',
    'Streaming',
    'Transporte/Bus',
    'Farmacia',
]
descriptions = {
    'Supermercado': ['Costco', 'Walmart', 'Soriana', 'Chedraui'],
    'Comida rápida': ['Burger King', 'McDonalds', 'Dominos', 'Uber Eats'],
    'Streaming': ['Amazon Prime', 'Netflix', 'Spotify', 'Disney+'],
    'Transporte/Bus': ['Bus', 'Uber', 'DiDi', 'Gasolinera'],
    'Farmacia': ['Farmacias Guadalajara', 'Farmacias del Ahorro'],
}

data = []
for i in range(n_samples):
  c_id = np.random.choice(clients)
  c_info = client_profiles[c_id]
  cat = np.random.choice(categories)
  desc = np.random.choice(descriptions[cat])

  if cat == 'Supermercado':
    monto = np.random.uniform(500, 3500)
  elif cat in ['Comida rápida', 'Streaming']:
    monto = np.random.uniform(80, 500)
  else:
    monto = np.random.uniform(20, 800)

  fecha = pd.Timestamp('2026-01-01') + pd.Timedelta(
      days=int(np.random.randint(0, 60)), hours=int(np.random.randint(0, 23))
  )

  data.append({
      'Id_Cliente': c_id,
      'Ingreso_Mensual_Cliente': c_info['ingreso'],
      'Ahorro_Actual_Cliente': c_info['ahorro'],
      'Fecha_Hora': str(fecha),
      'Tipo_Transaccion': 'Egreso',
      'Categoria_Transaccion': cat,
      'Descripcion_Transaccion': desc,
      'Cantidad_Monto': round(monto, 2),
      'Buro_Credito_Score': c_info['score'],
  })

df = pd.DataFrame(data)

# -------------------------------------------------------------
# MODELO 1: CLASIFICADOR DE CATEGORÍAS DE TRANSACCIÓN
# -------------------------------------------------------------
df['Fecha_Hora'] = pd.to_datetime(df['Fecha_Hora'])
df['es_fin_de_semana'] = np.where(df['Fecha_Hora'].dt.dayofweek >= 5, 1, 0)
df['ratio_gasto_ingreso'] = (
    df['Cantidad_Monto'] / df['Ingreso_Mensual_Cliente']
).round(4)

le_desc = LabelEncoder()
df['desc_encoded'] = le_desc.fit_transform(df['Descripcion_Transaccion'])

scaler_monto = StandardScaler()
df['monto_scaled'] = scaler_monto.fit_transform(df[['Cantidad_Monto']])

scaler_score = StandardScaler()
df['score_scaled'] = scaler_score.fit_transform(df[['Buro_Credito_Score']])

X_cat = df[[
    'monto_scaled',
    'desc_encoded',
    'es_fin_de_semana',
    'ratio_gasto_ingreso',
    'score_scaled',
]]
y_cat = df['Categoria_Transaccion']

modelo_cat = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_cat.fit(X_cat, y_cat)

# -------------------------------------------------------------
# MODELO 2: CLASIFICADOR DE SALUD FINANCIERA (PERFIL DEL CLIENTE)
# -------------------------------------------------------------
client_agg = (
    df.groupby('Id_Cliente')
    .agg({
        'Ingreso_Mensual_Cliente': 'first',
        'Ahorro_Actual_Cliente': 'first',
        'Buro_Credito_Score': 'first',
        'Cantidad_Monto': 'sum',
    })
    .reset_index()
)

client_agg['ratio_ahorro_ingreso'] = (
    client_agg['Ahorro_Actual_Cliente'] / client_agg['Ingreso_Mensual_Cliente']
)
client_agg['ratio_deuda_ingreso'] = (
    client_agg['Cantidad_Monto'] / client_agg['Ingreso_Mensual_Cliente']
)


# Reglas de negocio para etiquetar salud financiera
def clasificar_salud(row):
  if (
      row['ratio_deuda_ingreso'] > 0.45
      or row['Buro_Credito_Score'] < 580
      or row['ratio_ahorro_ingreso'] < 0.1
  ):
    return 'Riesgo Alto (Sobreendeudado)'
  elif row['ratio_ahorro_ingreso'] >= 0.3 and row['Buro_Credito_Score'] >= 700:
    return 'Saludable (Ahorrador)'
  else:
    return 'Equilibrado'


client_agg['Perfil_Salud_Financiera'] = client_agg.apply(
    clasificar_salud, axis=1
)

X_perfil = client_agg[[
    'Ingreso_Mensual_Cliente',
    'Ahorro_Actual_Cliente',
    'Buro_Credito_Score',
    'ratio_ahorro_ingreso',
    'ratio_deuda_ingreso',
]]
y_perfil = client_agg['Perfil_Salud_Financiera']

modelo_perfil = RandomForestClassifier(n_estimators=50, random_state=42)
modelo_perfil.fit(X_perfil, y_perfil)

# Guardar todos los artefactos en disco
joblib.dump(modelo_cat, 'modelo_categoria.pkl')
joblib.dump(le_desc, 'encoder_descripcion.pkl')
joblib.dump(scaler_monto, 'scaler_monto.pkl')
joblib.dump(scaler_score, 'scaler_score.pkl')
joblib.dump(modelo_perfil, 'modelo_perfil_salud.pkl')

print('Modelos y escaladores guardados correctamente.')

Modelos y escaladores guardados correctamente.


In [4]:
import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(
    title='API Salud Financiera y Clasificación de Gastos', version='1.0'
)

# Carga de modelos al iniciar la API
try:
  modelo_cat = joblib.load('modelo_categoria.pkl')
  le_desc = joblib.load('encoder_descripcion.pkl')
  scaler_monto = joblib.load('scaler_monto.pkl')
  scaler_score = joblib.load('scaler_score.pkl')
  modelo_perfil = joblib.load('modelo_perfil_salud.pkl')
except Exception as e:
  print(
      f'Error al cargar archivos .pkl. Asegúrate de ejecutar train_and_export.py primero: {e}'
  )


# --- Esquemas JSON ---
class TransaccionInput(BaseModel):
  Cantidad_Monto: float
  Ingreso_Mensual_Cliente: float
  Descripcion_Transaccion: str
  Buro_Credito_Score: int
  Fecha_Hora: str  # Formato: "YYYY-MM-DD HH:MM:SS"


class PerfilClienteInput(BaseModel):
  Ingreso_Mensual_Cliente: float
  Ahorro_Actual_Cliente: float
  Buro_Credito_Score: int
  Gastos_Totales_Mes: float


# --- Endpoint 1: Clasificación de Categoria de Gasto ---
@app.post('/predict-categoria')
def predecir_categoria(transaccion: TransaccionInput):
  try:
    fecha = pd.to_datetime(transaccion.Fecha_Hora)
    es_fin_de_semana = 1 if fecha.dayofweek >= 5 else 0
    ratio_gasto_ingreso = round(
        transaccion.Cantidad_Monto / transaccion.Ingreso_Mensual_Cliente, 4
    )

    try:
      desc_encoded = le_desc.transform([transaccion.Descripcion_Transaccion])[0]
    except ValueError:
      desc_encoded = 0  # Fallback si la descripción es totalmente desconocida

    monto_scaled = scaler_monto.transform([[transaccion.Cantidad_Monto]])[0][0]
    score_scaled = scaler_score.transform([[transaccion.Buro_Credito_Score]])[
        0
    ][0]

    X_in = np.array([[
        monto_scaled,
        desc_encoded,
        es_fin_de_semana,
        ratio_gasto_ingreso,
        score_scaled,
    ]])
    pred = modelo_cat.predict(X_in)[0]

    return {
        'status': 'success',
        'categoria_predicha': pred,
        'ratio_gasto_ingreso': ratio_gasto_ingreso,
    }
  except Exception as e:
    raise HTTPException(status_code=400, detail=str(e))


# --- Endpoint 2: Clasificación de Salud Financiera (Perfil) ---
@app.post('/predict-perfil-financiero')
def predecir_perfil_financiero(cliente: PerfilClienteInput):
  try:
    ratio_ahorro = (
        cliente.Ahorro_Actual_Cliente / cliente.Ingreso_Mensual_Cliente
    )
    ratio_deuda = (
        cliente.Gastos_Totales_Mes / cliente.Ingreso_Mensual_Cliente
    )

    X_in = np.array([[
        cliente.Ingreso_Mensual_Cliente,
        cliente.Ahorro_Actual_Cliente,
        cliente.Buro_Credito_Score,
        ratio_ahorro,
        ratio_deuda,
    ]])

    perfil_predicho = modelo_perfil.predict(X_in)[0]

    return {
        'status': 'success',
        'perfil_salud_financiera': perfil_predicho,
        'metricas': {
            'ratio_ahorro_ingreso': round(ratio_ahorro, 2),
            'ratio_gastos_ingreso': round(ratio_deuda, 2),
        },
    }
  except Exception as e:
    raise HTTPException(status_code=400, detail=str(e))